In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.5 MB/s eta 0:00:00


In [6]:
from my_preprocessing_classes import reduce_mem_usage

In [7]:
import pandas as pd
import mlflow.sklearn
import gc 

test_trans = reduce_mem_usage(pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv'))
test_id = reduce_mem_usage(pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv'))

test_raw = pd.merge(test_trans, test_id, on='TransactionID', how='left')

del test_trans, test_id
gc.collect() 

print(f"Memory optimized test set shape: {test_raw.shape}")

Memory usage of dataframe is 1519.24 MB


/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow e

Memory usage after optimization is: 425.24 MB
Decreased by 72.0%
Memory usage of dataframe is 44.39 MB


/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:


Memory usage after optimization is: 9.84 MB
Decreased by 77.8%
Memory optimized test set shape: (506691, 433)


In [14]:
import mlflow.sklearn
import pandas as pd

DAGSHUB_MLFLOW_URI = "https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow"
mlflow.set_tracking_uri(DAGSHUB_MLFLOW_URI)

model_name = "XGBoost_best_model"
model_version = 1 

model_uri = f"models:/{model_name}/{model_version}"
loaded_pipeline= mlflow.sklearn.load_model(model_uri)

print("მოდელი წარმატებით ჩაიტვირთა Registry-დან!")

მოდელი წარმატებით ჩაიტვირთა Registry-დან!


In [15]:
# import mlflow.sklearn
# import pandas as pd

# DAGSHUB_MLFLOW_URI = "https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow"
# mlflow.set_tracking_uri(DAGSHUB_MLFLOW_URI)

# model_name = "LightGBM_Best_model"
# model_version = 1 

# model_uri = f"models:/{model_name}/{model_version}"
# loaded_pipeline = mlflow.sklearn.load_model(model_uri)

# print("მოდელი წარმატებით ჩაიტვირთა Registry-დან!")

მოდელი წარმატებით ჩაიტვირთა Registry-დან!


In [9]:
fix_columns = {col: col.replace('-', '_') for col in test_raw.columns if '-' in col}
test_raw = test_raw.rename(columns=fix_columns)

In [13]:
test_preds = loaded_pipeline.predict(test_raw)
test_probs = loaded_pipeline.predict_proba(test_raw)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_raw['TransactionID'],
    'isFraud': test_probs
})

submission.to_csv('submission.csv', index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
